# MLP in Pytorch

Note: This notebook has been somewhat cut down with respect to the illustrations of the data set etc. to keep things a bit shorter. If you want the code to plot the data distribution, label distribution, plot images etc. check the linear model notebook!

In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from time import perf_counter

import numpy as np
import torch
from matplotlib import pyplot as plt

from idl.common import accuracy
from idl.week1.data import get_mnist, load_mnist, mnist_overview
from idl.week1.analysis import confusion_matrix, plot_learning_curves, precision_recall

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device is", DEVICE)

## Getting Data

In [ ]:
get_mnist()
(x_train, y_train), (x_valid, y_valid), (x_test, y_test) = load_mnist()

## Model

In [ ]:
class Model:
    def __init__(self, 
                 hidden_layer_sizes: list[int],
                 initialization_range: str | float = "glorot"):
        layer_sizes = hidden_layer_sizes + [10]  # add output layer
                 
        self.parameters = []
        previous_size = 784
        for layer_size in layer_sizes:
            if initialization_range == "glorot":
                weight_bound = np.sqrt(6) / np.sqrt(previous_size + layer_size)
            else:
                weight_bound = initialization_range
            layer_weights = [torch.distributions.Uniform(-weight_bound, weight_bound).sample((layer_size, previous_size)).requires_grad_(),
                             torch.zeros(layer_size, requires_grad=True)]
            previous_size = layer_size
            self.parameters.append(layer_weights)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        outputs = inputs
        for weights, bias in self.parameters[:-1]:
            outputs = outputs @ weights.transpose(0, 1) + bias
            outputs = torch.relu(outputs)
        # no hidden activation on final layer! this is the output layer
        weights, bias = self.parameters[-1]
        outputs = outputs @ weights.transpose(0, 1) + bias
        return log_softmax(outputs)

    def __call__(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.forward(inputs)

    def to(self, device: str):
        with torch.no_grad():
            for layer in self.parameters:
                for ind in range(len(layer)):
                    layer[ind] = layer[ind].to(device)
                    layer[ind].requires_grad = True
    
    @property
    def device(self):
        return self.parameters[0][0].device


def softmax(inputs: torch.Tensor) -> torch.Tensor:
    exponential = torch.exp(inputs)
    return exponential / exponential.sum(axis=-1, keepdims=True)


def log_softmax(inputs: torch.Tensor) -> torch.Tensor:
    return inputs - torch.logsumexp(inputs, dim=1, keepdims=True)


def cross_entropy(log_outputs: torch.Tensor,
                  labels: torch.Tensor) -> torch.Tensor:
    labels_one_hot = one_hot(labels, num_classes=10)
    return -(labels_one_hot * log_outputs).sum(axis=-1).mean()


def one_hot(inputs: torch.Tensor,
            num_classes: int | None = None) -> torch.Tensor:
    if num_classes is None:
        num_classes = inputs.max() + 1
    output = torch.zeros(inputs.shape[0], num_classes, device=inputs.device)
    output.scatter_(1, inputs.view(-1, 1), torch.ones(inputs.shape[0], 1, device=inputs.device))

    return output

In [ ]:
mlp_model = Model([512])
with torch.no_grad():
    print(mlp_model(torch.tensor(x_train[:4])))

In [ ]:
# NOTE! if running on GPU, after running this cell, 
# the above tests will no longer work because the weights are on GPU,
# but the input tensors on CPU.
# you would have to move the input tensors to GPU a well!
mlp_model.to(DEVICE)
print(mlp_model.parameters[0][0].device)

## Training

Again, very little changes here! We just adapted the code for applying the gradients a little bit to accommodate hidden layers.

In [ ]:
def train_model(model: Model, 
                training_data: tuple[np.ndarray, np.ndarray], 
                validation_data: tuple[np.ndarray, np.ndarray],
                learning_rate: float,
                batch_size: int,
                n_epochs: int,
                verbose: bool = True):
    n_training_examples = len(training_data[0])
    batches_per_epoch = n_training_examples // batch_size
    print("Running {} epochs at {} steps per epoch.".format(n_epochs, batches_per_epoch))
    
    random_shuffling = np.arange(n_training_examples)

    train_input_tensor = torch.tensor(training_data[0]).to(model.device)
    train_label_tensor = torch.tensor(training_data[1]).to(model.device)
    valid_input_tensor = torch.tensor(validation_data[0]).to(model.device)
    valid_label_tensor = torch.tensor(validation_data[1]).to(model.device)

    # note, for training we only track the average over the epoch.
    # this is somewhat imprecise, as the model changes over the epoch.
    # so the metrics at the end of the epoch will usually be better than at the start,
    # but we average over everything.
    # we could record train metrics more often to get a better picture of training progress.
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(n_epochs):
        if verbose:
            print("Starting epoch {}...".format(epoch + 1), end=" ")
        start_time = perf_counter()
        # every epoch we shuffle the data!
        np.random.shuffle(random_shuffling)  # this shuffles in place
        slice_index = 0
        shuffled_inputs = train_input_tensor[random_shuffling]
        shuffled_labels = train_label_tensor[random_shuffling]

        epoch_train_losses = []
        epoch_train_accuracies = []
        
        for batch_ind in range(batches_per_epoch):
            input_batch = shuffled_inputs[slice_index:slice_index+batch_size]
            label_batch = shuffled_labels[slice_index:slice_index+batch_size]
            # this is the core of model training!
            # all the other stuff is pretty much just providing data and recording metrics...
            # two lines "forward" to compute outputs and the loss
            output_batch = model(input_batch)
            batch_loss = cross_entropy(output_batch, label_batch)
            # and then backpropagate
            batch_loss.backward()
            with torch.no_grad():
                for layer in model.parameters:
                    for parameter in layer:
                        parameter.sub_(parameter.grad*learning_rate)
                        parameter.grad.zero_()

                batch_accuracy = accuracy(output_batch, label_batch)
                epoch_train_losses.append(batch_loss.item())
                epoch_train_accuracies.append(batch_accuracy.item())
            
            slice_index += batch_size
        end_time = perf_counter()
        time_taken = end_time - start_time
            
        # evaluate after each epoch
        with torch.no_grad():
            val_output = model(valid_input_tensor)
            val_loss = cross_entropy(val_output, valid_label_tensor)
            val_accuracy = accuracy(val_output, valid_label_tensor)
            val_losses.append(val_loss.item())
            val_accuracies.append(val_accuracy.item())

        train_losses.append(np.mean(epoch_train_losses))
        train_accuracies.append(np.mean(epoch_train_accuracies))

        if verbose:
            print(f"Time taken: {time_taken:.3f} seconds")
            print(f"\tTrain/val loss: {train_losses[-1]:.5g} / {val_losses[-1]:.5g}")
            print(f"\tTrain/val accuracy: {train_accuracies[-1]:.5g} / {val_accuracies[-1]:.5g}")
            print()
        
    return {"train_loss": np.array(train_losses), "train_accuracy": np.array(train_accuracies),
            "val_loss": np.array(val_losses), "val_accuracy": np.array(val_accuracies)}

In [ ]:
metrics = train_model(mlp_model, (x_train, y_train), (x_valid, y_valid),
                      learning_rate=0.3, batch_size=128, n_epochs=15)

## Inspecting Model Performance

In [ ]:
plot_learning_curves(metrics, keys=["loss", "accuracy"])

This is the same confusion matrix code from the linear models.

In [ ]:
training_matrix = confusion_matrix(mlp_model, x_train, y_train, device=DEVICE)
np.set_printoptions(suppress=True)
print("Training confusion matrix")
print(training_matrix)

validation_matrix = confusion_matrix(mlp_model, x_valid, y_valid, device=DEVICE)
print()
print("Validation confusion matrix")
print(validation_matrix)

In [ ]:
print("TRAINING")
precision_recall(training_matrix)

print("\n\nVALIDATION")
precision_recall(validation_matrix)

## Inspecting Model Behavior

Just like in a linear model, we can try to plot the weights as images. However, this only makes sense for the first hidden layer, as only that one receives the images directly. We could still display the deeper weights in a 2D shape, but that wouldn't really make sense, as the inputs to those layers are not images anymore, but arbitrary feature vectors returned by the earlier layers.

As such, we can plot the features learned by the first layer, which would then be further processed by the higher layers. These most often seem to be little edges or curved segments, although the features tend to be quite noisy. In this case, some regularization of the weights would probably lead to more interpretable features.

We have a choice of using one colormap per feature, or the same colormap for all features. The former shows as much detail as possible for every feature, but it can be misleading, as very small weights for one feature could look as significant as larger weights for another. The second option gives us a better impression of which features are "stronger' than others.

In [ ]:
def visualize_features(colormap="local"):
    if colormap not in ["local", "global"]:
        raise ValueError("colormap argument should be 'local' or 'global'")
    features = mlp_model.parameters[0][0].detach().cpu().numpy()
    global_absmax = abs(features).max()
    
    plt.figure(figsize=(12, 6))
    for ind, pattern in enumerate(features):
        plt.subplot(16, 32, ind+1)
        absmax = abs(pattern).max() if colormap == "local" else global_absmax
        plt.imshow(pattern.reshape(28, 28), vmin=-absmax, vmax=absmax, cmap="RdBu_r")
        plt.axis("off")
        #plt.colorbar()
    plt.suptitle(f"First layer features with {colormap} colormaps")
    plt.show()

In [ ]:
visualize_features("local")
visualize_features("global")

By inspecting the biases, we can look at what labels the model "prefers" absent any input data. However, these values are hard to interpret without the weights in a realistic scenario. For example, the largest bias is for class 5, yet this is actually the rarest class in the dataset, and so it should have the lowest score a priori. Note that these values will differ based on network architecture, training process and random factors.

In [ ]:
plt.bar(np.arange(10), mlp_model.parameters[-1][1].detach().cpu().numpy())
plt.xticks(np.arange(10))
plt.title("Bias for each label")
plt.show()

## Dead ReLUs

In the feature plot above, you may (depending on how training went) some weights that just look like random noise. It seems as if those units haven't really learned anything. How can that be?

With ReLU activations, it may happen that a unit _always_ returns negative outputs with the linear mapping, which are then rectified to 0 by ReLU. This cuts off the gradients and prevents those units from ever learning anything. They are useless!

To check this, we can apply only the hidden layer to the inputs, and then, for each dimension/feature, check the proportion of values below 0.

In [ ]:
with torch.no_grad():
    training_outputs_hidden =  (torch.tensor(x_train).to(DEVICE) @ mlp_model.parameters[0][0].transpose(0, 1) 
                                + mlp_model.parameters[0][1])
    proportion_negative = (training_outputs_hidden <= 0.).to(torch.float32).mean(axis=0)

In [ ]:
props_sorted = sorted(proportion_negative.cpu().numpy(), reverse=True)
plt.plot(props_sorted)
plt.xlabel("Unit index (sorted)")
plt.ylabel("Proportion of negative activations in training data")
plt.show()

In [ ]:
# by looking at the least active units, we can see if there are any with a proportion of 1 (always inactive).
# in my case this wasn't true for any, but there were some pretty close ones.
# these units will be almost never used, and thus receive barely any gradient updates.
props_sorted[:10]

## Basic Hyperparameter Study: Learning Rate

This is like the linear model example.

In [ ]:
# to cover a large space, we commonly use so-called geometric progressions.
# this means there is a common *multipilicative* factor between successive values.
# here, each learning rate is roughly 3x the one before.
learning_rates = np.geomspace(0.0000001, 10., 17)
learning_rates

In [ ]:
all_metrics = {}
for lr_ind, learning_rate in enumerate(learning_rates):
    model = Model([512])
    model.to(DEVICE)
    print(f"Training learning rate {learning_rate}...")
    metrics = train_model(model, (x_train, y_train), (x_valid, y_valid),
                          learning_rate=learning_rate, batch_size=128, n_epochs=15,
                          verbose=False)
    all_metrics[lr_ind] = metrics

In [ ]:
# collect the final results only
final_metrics = {"val_loss": [], "train_loss": [], "val_accuracy": [], "train_accuracy": []}
for ind in range(len(learning_rates)):
    lr_results = all_metrics[ind]
    for key in lr_results:
        final_metrics[key].append(lr_results[key][-1])

In [ ]:
plt.plot(final_metrics["train_loss"], label="train")
plt.plot(final_metrics["val_loss"], label="validation")
plt.legend()
plt.title("Loss")
plt.xticks(np.arange(len(learning_rates)), np.round(learning_rates, 7), rotation="vertical")
plt.xlabel("Learning rate")
plt.show()

plt.plot(final_metrics["train_accuracy"], label="train")
plt.plot(final_metrics["val_accuracy"], label="validation")
plt.legend()
plt.title("Accuracy")
plt.xticks(np.arange(len(learning_rates)), np.round(learning_rates, 7), rotation="vertical")
plt.xlabel("Learning rate")
plt.show()

best_loss_index = np.argmin(final_metrics["val_loss"])
print(f"Best validation loss: {final_metrics["val_loss"][best_loss_index]} "
      f"for learning rate {learning_rates[best_loss_index]}")

best_accuracy_index = np.argmax(final_metrics["val_accuracy"])
print(f"Best validation accuracy: {final_metrics["val_accuracy"][best_accuracy_index]} "
      f"for learning rate {learning_rates[best_accuracy_index]}")

In [ ]:
for key in lr_results:
    for ind, lr in enumerate(learning_rates):
        lr_results = all_metrics[ind]
        plt.plot(lr_results[key], label=str(np.round(lr, 7)))
    plt.title(key)
    plt.legend()
    plt.xlabel("Epoch")
    plt.show()

## Number of Hidden Units

We can also look at how increasing the number of hidden units affects the performance. This is a good time to bring up an important point: Training a model multiple times can lead to different results, and significantly different quality models. The amount of variance can depend on the architecture, training hyperparameters like learning rate or batch size, etc. Unfortunately, it's rarely feasible to re-train our models multiple times. But for simple MLPs, we can afford to do it, so let's try.

In [ ]:
n_units_options = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024, 1536, 2048]
n_reruns = 20

all_metrics_units = {}
for n_units in n_units_options:
    print("Trying with {} hidden units...".format(n_units))
    all_metrics_units[n_units] = []
    for run in range(n_reruns):
        print("Run", run+1)
        model = Model([n_units])
        model.to(DEVICE)
        metrics = train_model(model, (x_train, y_train), (x_valid, y_valid),
                              learning_rate=0.03, batch_size=128, n_epochs=15,
                              verbose=False)
        all_metrics_units[n_units].append(metrics)

In [ ]:
# collect the final results only
final_metrics_units = {"val_loss": {}, "train_loss": {}, "val_accuracy": {}, "train_accuracy": {}}
for n_units in n_units_options:
    for key in final_metrics_units:
        final_metrics_units[key][n_units] = []
        
    unit_results = all_metrics_units[n_units]
    for run in unit_results:
        for key in run:
            final_metrics_units[key][n_units].append(run[key][-1])

We can see that performance is generally quite consistent. For very small numbers of hidden units, there is a bit more variance. This could be explained by the use of the `relu` activation -- depending on luck with the initialization, if some of the very small number of units are mostly or always inactive, this has a much larger negative influence than if we have many units available.

In [ ]:
for key in final_metrics_units:
    key_results = final_metrics_units[key]
    means = []
    for n_units in n_units_options:
        unit_results = key_results[n_units]
        x = n_units * np.ones_like(unit_results)
        x += np.random.uniform(-0.05*n_units, 0.05*n_units, x.shape)
        plt.scatter(x, unit_results, c="black", alpha=0.1, marker=".")
        means.append(np.array(unit_results).mean())
    plt.plot(n_units_options, means, "k--")

    plt.title(key)
    plt.xlabel("Number of hidden units")
    plt.xscale("log")
    plt.show()

## Interactions

Finally, we can see what happens when we change two hyperparameters together which may be related in some way. Since the question came up in one of the exercises, we will look at the batch size a bit more. Increasing the batch size leads to more consistent gradient updates, but also fewer steps per epoch. So we may have to increase the number of epochs to compensate. So here we sweep over a range of batch sizes and number of epochs to see if there are any interaction effects. We also train multiple models for each setting to protect against "bad luck" outcomes as we've seen before.

We only check a limited range of hyperparameter values. For example, we could go down to a batch size of 1. But that leads to 50,000 steps for a single epoch, and then combining this with many more epochs would take a *very* long time. To offset this, we do not run any configurations with more than 50,000 steps, replacing the metrics with `nan`. This could be slightly confusing in cases where the training is actually unstable and results in `nan` as well. :)

This takes VERY long to train, like a day or more on a recent GPU.

In [ ]:
n_reruns = 10

n_epochs_options = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
batch_size_options = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

nan_metrics = {"train_loss": np.array([np.nan]), "val_loss": np.array([np.nan]),
               "train_accuracy": np.array([np.nan]), "val_accuracy": np.array([np.nan])}

learning_rates = [0.0003, 0.003, 0.03, 0.3, 3.]
lr_interaction_final_results = {}

print("Training {} models overall".format(
    len(learning_rates) * len(n_epochs_options) * len(batch_size_options) * n_reruns))
for lr_ind, lr in enumerate(learning_rates):
    print("Learning rate:", lr)
    all_metrics_interaction = {}
    for n_epochs in n_epochs_options:
        all_metrics_interaction[n_epochs] = dict()
        print("Trying with {} epochs...".format(n_epochs))
        for batch_size in batch_size_options:
            all_metrics_interaction[n_epochs][batch_size] = []
            print("\tTrying with batch size {}...".format(batch_size))
            for run in range(n_reruns):
                if n_epochs * (50000 // batch_size) > 50000:
                    metrics = nan_metrics
                else:
                    print("\t\tRun", run+1)
                    model = Model([n_units])
                    model.to(DEVICE)
                    metrics = train_model(model, (x_train, y_train), (x_valid, y_valid),
                                          learning_rate=lr, batch_size=batch_size, n_epochs=n_epochs,
                                          verbose=False)
                all_metrics_interaction[n_epochs][batch_size].append(metrics)

    # collect the final results only
    from collections import defaultdict
    final_metrics_interaction = {"val_loss": defaultdict(dict),
                                  "train_loss": defaultdict(dict),
                                  "val_accuracy": defaultdict(dict),
                                  "train_accuracy": defaultdict(dict)}
    
    for n_epochs in n_epochs_options:
        for batch_size in batch_size_options:
            for key in final_metrics_interaction:
                final_metrics_interaction[key][n_epochs][batch_size] = []
            interaction_results = all_metrics_interaction[n_epochs][batch_size]
            for run in interaction_results:
                for key in run:
                    final_metrics_interaction[key][n_epochs][batch_size].append(run[key][-1])

    lr_interaction_final_results[lr_ind] = final_metrics_interaction

In [ ]:
# collect the final results only
from collections import defaultdict
final_metrics_interaction = {"val_loss": defaultdict(dict),
                              "train_loss": defaultdict(dict),
                              "val_accuracy": defaultdict(dict),
                              "train_accuracy": defaultdict(dict)}

for n_epochs in n_epochs_options:
    for batch_size in batch_size_options:
        for key in final_metrics_interaction:
            final_metrics_interaction[key][n_epochs][batch_size] = []
        interaction_results = all_metrics_interaction[n_epochs][batch_size]
        for run in interaction_results:
            for key in run:
                final_metrics_interaction[key][n_epochs][batch_size].append(run[key][-1])

In [ ]:
for lr_ind, lr in enumerate(learning_rates):
    print("\n\n\nLearning rate", lr)
    final_metrics_interaction = lr_interaction_final_results[lr_ind]
    for key in final_metrics_interaction:
        key_results = final_metrics_interaction[key]
        means = np.zeros((len(n_epochs_options), len(batch_size_options)))
        for epoch_ind, n_epochs in enumerate(n_epochs_options):
            for batch_size_ind, batch_size in enumerate(batch_size_options):
                interaction_results = key_results[n_epochs][batch_size]
                means[epoch_ind, batch_size_ind] =  np.array(interaction_results).mean()

        if "loss" in key:
            vmin, vmax = 0., np.log(10)
        elif "accuracy" in key:
            vmin, vmax = 0., 1.
        plt.imshow(means, vmin=vmin, vmax=vmax)
    
        plt.title(key)
        plt.xlabel("Batch size")
        plt.ylabel("Number of epochs")
        plt.colorbar()
    
        plt.xticks(np.arange(len(batch_size_options)), batch_size_options)
        plt.yticks(np.arange(len(n_epochs_options)), n_epochs_options)
        
        plt.show()